# Chapter 8: System Design and Engineering Judgment

Estimated time: ~5 hours.

Prerequisites: Chapters 1-7. This chapter is a synthesis, not a new technique. Every
question in its framework points back at a chapter you have already built real code for.

## Setup

This chapter writes more than it runs -- the artifact is a design doc, not a system --
but the two graded cells still need the autograder.

In [1]:
import sys
from pathlib import Path

# Make `agentlib` importable regardless of this notebook's working directory.
_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from agentlib.grading import check

print(f"Repo root on sys.path: {_repo_root}")
print("This chapter makes no model calls -- the work is written, not executed.")

Repo root on sys.path: /home/user/learning-agentic-ai
This chapter makes no model calls -- the work is written, not executed.


## Section 1: Definitions

### 9-question design framework

A system-design interview question ("design an agent that does X") tests whether you can
walk through a design in a repeatable order, out loud, without skipping to architecture
before establishing whether an agent is even the right tool. Nine questions, in order:

| # | Question | Chapter |
|---|---|---|
| 0 | Does this actually need an agent? | Ch 8 |
| 1 | What does success look like, and what does a failure cost? | -- |
| 2 | What control flow fits: single call, fixed workflow, ReAct, or multi-agent? | Ch 1-2 |
| 3 | Does it need retrieval/grounding, and against what corpus? | Ch 3 |
| 4 | What is the reliability story: caching, retries, circuit breakers? | Ch 4 |
| 5 | What is the cost/latency budget, and does model routing help? | Ch 5 |
| 6 | What is the security surface: untrusted content, tool privilege? | Ch 6 |
| 7 | What tools does it need, and what happens when one fails? | Ch 7 |
| 8 | How will you know it is working, in production, over time? | Ch 3, Ch 9 |

The order matters. Question 0 comes first on purpose: jumping straight to "here is my
architecture" without asking whether an agent is the right call at all is one of the most
common tells that someone has not actually shipped one.

### When NOT to use an agent

Five concrete reasons:

1. **The task is deterministic.** If the "decision" is really `if/elif/else` over
   well-understood cases, plain code is faster, cheaper, and does not need Chapter 6's
   entire threat model. An agent adds nondeterminism and a security surface to a problem
   that had neither.

2. **A single LLM call suffices.** Summarization, classification, extraction, and
   rewriting tasks are often genuinely single-shot. No multi-step decision means no agent
   to build.

3. **High-stakes, high-volume, irreversible.** If the failure cost is high and the volume
   is high enough that a small error rate is unacceptable at scale, and no verification
   step can catch it before it causes harm, an agent's probabilistic nature is a bad fit
   unless paired with a hard gate (Chapter 6's policy-check patterns).

4. **Latency requirements are too tight.** If the budget is tens of milliseconds, no amount
   of model routing (Chapter 5) closes the gap between that and an LLM round-trip.

5. **You cannot define success.** If there is no way to tell whether the agent did the
   right thing (Chapter 3's evaluation metrics, or even a simpler pass/fail check), you
   cannot safely iterate on it in production either.

Real-world examples:
- **Data pipeline ETL** (deterministic): fixed transformations, no judgment needed.
- **Sentiment classification** (single call): one prompt, one label, no loop.
- **Medical prescription** (high-stakes irreversible): wrong dose at scale is catastrophic.
- **Real-time ad bidding** (latency): 10ms budget, no room for an LLM round-trip.

### Design document structure

A `DesignDoc` is a communication tool, not just code. Each of its 9 fields maps to one
framework question. Grading is structural: all fields answered, sufficient depth (40+ words
each), no placeholders, no duplicates. There is no single correct design for any scenario.

## Section 2: Concept Explanation

### The framework as a decision sequence

```
  "Design an agent that does X"
       |
       v
  Q0: Does it need an agent at all?
       |
      NO --> What simpler thing do you build instead?
       |      (single LLM call, workflow, rule engine)
      YES
       |
       v
  Q1: What does success look like?
      What does a failure cost?
       |
       v
  Q2: What control flow?
      (single call / workflow / ReAct / multi-agent)
       |
       v
  Q3-Q7: Retrieval, reliability, cost, security, tools
         (each constrained by Q1 and Q2)
       |
       v
  Q8: How do you know it is working?
      (offline metrics + online monitoring)
```

The first two questions constrain everything that follows. A workflow that fails safely
does not need the same reliability investment as an autonomous agent making irreversible
calls. A system whose failure cost is "a human reviews it anyway" has different security
requirements than one where the agent acts autonomously.

### Trade-offs: over-engineering vs. under-engineering

| Signal | Risk | Example |
|---|---|---|
| Building an agent for a deterministic task | Over-engineering | ETL pipeline with an LLM |
| Building a workflow for a task that needs judgment | Under-engineering | Support triage with hard-coded routing |
| Skipping reliability for a prototype | Acceptable for v0 | But document what you skipped |
| Skipping security for any user-facing system | Never acceptable | Chapter 6 applies from day one |

### Three worked studios

This chapter provides three design-doc scenarios, each exercising a different mix of the
nine questions:

1. **Support-ticket triage agent**: close to Chapter 6's mock system, but designing the
   *system* around it rather than attacking it. Heavy on security (Q6) and reliability (Q4).
2. **PR review assistant**: a tool that drafts review comments for a human to accept or
   dismiss. Heavy on evaluation (Q8) and cost (Q5).
3. **Legal research assistant**: answers questions over internal contracts, citing sources.
   Heavy on retrieval (Q3) and trust/failure cost (Q1).

## Section 3: Example Code Segments

The `DesignDoc` dataclass and the three studio scenarios.

### The `DesignDoc` dataclass

Nine fields, one per framework question. `blank_fields()` reports which ones you have
not filled in yet.

In [3]:
from dataclasses import dataclass, field, fields


@dataclass
class DesignDoc:
    scenario: str
    needs_an_agent: str = ""          # question 0
    success_and_failure_cost: str = ""  # question 1
    control_flow: str = ""            # question 2 (Ch 1-2)
    retrieval_and_grounding: str = ""  # question 3 (Ch 3)
    reliability_story: str = ""       # question 4 (Ch 4)
    cost_latency_budget: str = ""     # question 5 (Ch 5)
    security_surface: str = ""        # question 6 (Ch 6)
    tools_and_failure_modes: str = ""  # question 7 (Ch 7)
    evaluation_plan: str = ""         # question 8 (Ch 3, Ch 9)

    def blank_fields(self) -> list[str]:
        return [f.name for f in fields(self) if f.name != "scenario" and not getattr(self, f.name)]


blank = DesignDoc(scenario="(fill this in per studio below)")
print(f"Framework fields to complete: {blank.blank_fields()}")

Framework fields to complete: ['needs_an_agent', 'success_and_failure_cost', 'control_flow', 'retrieval_and_grounding', 'reliability_story', 'cost_latency_budget', 'security_surface', 'tools_and_failure_modes', 'evaluation_plan']


### Design-doc studios

Three scenarios. For each, work through all nine questions from the framework above on
your own before checking the solutions file. The studios are deliberately blank here;
fully worked versions live only in the solutions file, exactly like every other
chapter's interview-drill answers.

### Studio 1: a support-ticket triage agent

A company wants an agent that reads incoming support tickets, answers simple questions from
their help-center docs, and can issue refunds up to a policy-defined limit without a human in
the loop. (This is deliberately close to Chapter 6's mock system: designing the *system*
around it, this time, not attacking it.)

Work through all nine questions:

0. Does this actually need an agent?
1. What does success look like, and what does a failure cost?
2. What control flow fits?
3. Does it need retrieval/grounding, and against what corpus?
4. What's the reliability story?
5. What's the cost/latency budget?
6. What's the security surface?
7. What tools does it need, and what happens when one fails?
8. How will you know it's working, in production, over time?

### Studio 2: a codebase-aware pull-request review assistant

An engineering team wants a tool that, given a pull request's diff, drafts review comments
(flagging likely bugs, style inconsistencies, and missing tests) for a human reviewer to
accept, edit, or dismiss before anything is posted.

Work through all nine questions (same list as Studio 1).

### Studio 3: a research assistant over a large internal document set

A legal team wants a tool that answers questions about a large corpus of internal contracts
and policy documents, citing which document(s) each answer came from, for a team that will
lose trust in the tool immediately if it ever states something confidently that isn't
actually in the source documents.

Work through all nine questions (same list as Studio 1).

## Section 4: Build It Yourself

Two graded tasks: articulating when NOT to use an agent, and filling in a complete
design document for one of the three studios above.

### Task 1: `MY_NO_AGENT_CASE` (when not to use an agent)

Close the Definitions section above before writing this. The check wants at least two
of the five reasons, named distinctly, plus what you would build instead -- because
"don't use an agent" on its own leaves the interviewer guessing whether you have an
alternative in mind or just a reservation.

In [ ]:
# At least two of the five reasons, in your own words, and what you'd build instead.
MY_NO_AGENT_CASE = """Replace this with your answer."""


MY_NO_AGENT_CASE = check("ch08-no-agent-case", MY_NO_AGENT_CASE)

### Task 2: `MY_DESIGN_DOC` (complete design document)

Pick ONE of the three studios above and fill in all nine fields. Not notes -- sentences,
the way you would say them out loud in the room.

Grading is structural: all nine fields answered, each at least 40 words, no
placeholders, no duplicates. There is no single correct design; the check enforces that
you actually engaged with every question at depth.

In [ ]:
# Pick ONE of the three studios below and fill in all nine fields for it. Not notes --
# sentences, the way you would say them out loud in the room.
MY_DESIGN_DOC = DesignDoc(
    scenario="",
    needs_an_agent="",
    success_and_failure_cost="",
    control_flow="",
    retrieval_and_grounding="",
    reliability_story="",
    cost_latency_budget="",
    security_surface="",
    tools_and_failure_modes="",
    evaluation_plan="",
)


MY_DESIGN_DOC = check("ch08-design-doc", MY_DESIGN_DOC)

## Section 5: Playground

Experiments applying the 9-question framework to different scenarios.

### Experiment 1: Apply the framework to a chatbot use case

A product team asks for a customer-facing chatbot that answers FAQ questions from a
knowledge base. Walk through the first three questions and decide: agent, workflow,
or single LLM call?

In [ ]:
# --- WRITE YOUR ANALYSIS ---
# Q0: Does this need an agent?
#   (Consider: is there multi-step decision-making, or is it retrieve-and-answer?)
#
# Q1: What does success look like? What does failure cost?
#   (Consider: wrong answer to a customer vs. "I don't know")
#
# Q2: What control flow?
#   (Consider: single RAG call vs. ReAct loop vs. workflow)

chatbot_analysis = DesignDoc(
    scenario="Customer-facing FAQ chatbot over a knowledge base",
    needs_an_agent="",  # fill this in
    success_and_failure_cost="",  # fill this in
    control_flow="",  # fill this in
)
print(f"Blank fields remaining: {chatbot_analysis.blank_fields()}")

### Experiment 2: Apply the framework to a code review use case

Same framework, different scenario: an internal tool that drafts PR review comments.
How do the answers to Q1 (failure cost) and Q6 (security) change compared to the
chatbot?

In [ ]:
# --- WRITE YOUR ANALYSIS ---
# Compare: chatbot failure cost vs. code review failure cost
# Compare: chatbot security surface vs. code review security surface

code_review_analysis = DesignDoc(
    scenario="Internal PR review assistant that drafts comments for human review",
    needs_an_agent="",  # fill this in
    success_and_failure_cost="",  # fill this in
    security_surface="",  # fill this in
)
print(f"Blank fields remaining: {code_review_analysis.blank_fields()}")

### Experiment 3: Vary the failure cost

Take the support-ticket triage scenario from Studio 1. What changes in your design if
the failure cost shifts from "a human reviews it anyway" to "the agent can issue
refunds up to $10,000 with no human in the loop"?

In [ ]:
# --- WRITE YOUR ANALYSIS ---
# Low failure cost: human reviews every agent decision
# High failure cost: agent acts autonomously on refunds up to $10,000
#
# Which framework questions change their answer?
# (Hint: Q4 reliability, Q6 security, and Q8 evaluation all shift)

print("Questions whose answer changes when failure cost increases:")
print("  Q4 (reliability): ...")
print("  Q6 (security): ...")
print("  Q8 (evaluation): ...")

## Section 6: Break It

A team built an agent for a task that should have been a workflow. The agent costs 10x
more, has 3x more failure modes, and is slower.

**Scenario**: An e-commerce company built an LLM agent to process returns. The agent reads
the return request, looks up the order, checks the return policy, and issues the refund.
Every run follows the exact same sequence. The "decisions" are all deterministic: is the
item within the return window? Is it in returnable condition? Is the refund amount correct?

The agent costs $0.15 per return (LLM calls). The deterministic workflow it replaced cost
$0.002 per return (database lookups). The agent is also slower (2-3 seconds vs. 200ms) and
occasionally hallucinates a policy that does not exist, approving returns that should have
been denied.

**Hint 1**: Check the first design question: does this need dynamic control flow?

**Hint 2**: If every run follows the same sequence, it is a workflow, not an agent.

**Production impact**: 75x cost increase, 10-15x latency increase, and a new failure mode
(hallucinated policy) that the deterministic system could not have produced.

**Interview follow-up**: "How do you convince a team that has already built an agent that
they should switch to a simpler architecture?"

In [ ]:
# The over-engineered returns agent vs. the workflow it should have been
#
# Agent path (what they built):
#   LLM reads request -> LLM looks up order -> LLM checks policy -> LLM issues refund
#   Cost: $0.15/return, Latency: 2-3s, Failure modes: hallucination, timeout, cost
#
# Workflow path (what they should have built):
#   parse_request() -> lookup_order(db) -> check_policy(rules) -> issue_refund(api)
#   Cost: $0.002/return, Latency: 200ms, Failure modes: DB timeout (retryable)

agent_cost_per_return = 0.15
workflow_cost_per_return = 0.002
daily_returns = 5000

print(f"Daily cost comparison at {daily_returns} returns/day:")
print(f"  Agent:    ${agent_cost_per_return * daily_returns:,.2f}/day")
print(f"  Workflow: ${workflow_cost_per_return * daily_returns:,.2f}/day")
print(f"  Ratio:    {agent_cost_per_return / workflow_cost_per_return:.0f}x")
print()
print("The agent added:")
print("  - Nondeterminism (hallucinated policies)")
print("  - A security surface (prompt injection through return requests)")
print("  - 10-15x latency")
print("  - 75x cost")
print("  - Zero new capability (every decision was deterministic)")

## Section 7: Interview Q&A

### Question 1: "Walk me through how you would design an AI agent system from scratch."

**Model answer**: Start with Question 0: does this actually need an agent? If the task is
deterministic, a single LLM call, or has latency requirements an LLM cannot meet, stop
there and say what you would build instead. If it does need an agent, establish the failure
cost (Q1) and control flow (Q2) before anything else, because those two constrain every
later decision. Then work through retrieval (Q3), reliability (Q4), cost (Q5), security
(Q6), and tools (Q7) in order. End with evaluation (Q8): how you will know it is working
in production. The order matters because each question's answer constrains the next one.

### Question 2: "When would you NOT use an agent?"

**Model answer**: Five cases. (1) The task is deterministic: `if/elif/else` is faster,
cheaper, and does not introduce a security surface. (2) A single LLM call suffices:
summarization, classification, extraction. (3) High-stakes, high-volume, irreversible, with
no verification gate. (4) Latency budget is tighter than an LLM round-trip. (5) You cannot
define what "success" looks like well enough to evaluate it. For each one, name the simpler
thing you would build instead: a rule engine, a single prompt, a workflow with a human gate,
a cached lookup, or "step back and define the evaluation criteria first."

### Question 3: "How do you evaluate whether an agent system is working in production?"

**Model answer**: Two tracks. Offline: build an evaluation set with known-good answers and
measure precision, recall, and MRR (Chapter 3) on a regular cadence. Online: canary releases
(Chapter 9) route a small percentage of traffic to the new version and compare drift metrics
against baseline. The gap between offline and online metrics is itself a signal: if offline
looks great but production quality is poor, the evaluation set does not represent real
traffic. Specific metrics depend on the task: for support triage, measure resolution rate
and escalation rate; for retrieval, measure retrieval precision at the k used in production.

### Question 4: "A team wants to add an agent to their pipeline. What questions do you ask first?"

**Model answer**: Three questions before anything about architecture. First: what does the
agent decide that is not already determined by existing logic? If the answer is "nothing,"
the team wants an LLM for the wrong reason. Second: what happens when the agent is wrong?
If the failure is irreversible and there is no human check, the risk calculus changes.
Third: how will you know the agent is performing well? If there is no evaluation plan, the
team will ship something they cannot monitor, iterate on, or safely roll back.

### System design drill

Answer each of these on your own before checking the model answers.

In [5]:
from agentlib.self_check import drill as open_drill

drill = open_drill(8)
drill.questions()

Chapter 8 written drill — 4 questions

1. "Just add an agent" to a deterministic pipeline
2. Justifying multiple reliability/security layers without "piling on techniques"
3. Cold: "design an agent that restarts failed cloud services automatically"
4. Listing techniques vs. demonstrating judgment


#### Answering these

Write your answer into the slot for each question, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side. `check(n)` will
not show you an answer until you have written one of your own. If you want it anyway,
`drill.reveal(n)` is there and makes no judgement.

In [6]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 8: 0/4 answered
  still open: [1, 2, 3, 4]


In [7]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  "Just add an agent" to a deterministic pipeline

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References

1. All prior chapters: each maps to one or more framework questions.
   - Ch 1-2: control flow (Q2)
   - Ch 3: retrieval and evaluation (Q3, Q8)
   - Ch 4: reliability (Q4)
   - Ch 5: cost and latency (Q5)
   - Ch 6: security (Q6)
   - Ch 7: tool integration (Q7)

2. System design interview resources:
   - Xu, A. (2022). "System Design Interview" (Vol. 1 and 2).
   - Kleppmann, M. (2017). "Designing Data-Intensive Applications."

3. Agent design patterns:
   - Anthropic (2024). "Building effective agents."
   - LangChain / LangGraph documentation on agent architectures.

Related chapters:
- Chapter 9 (deployment, canary releases, drift detection -- the "Q8" answer in practice)
- `interview_prep/` (the capstone mock interview uses this framework directly)

## Next: Chapter 9, LLMOps and Deployment

This chapter did not introduce a new technique. It introduced an order to ask questions
in, tying every prior chapter's technique to the specific design question it answers.
Chapter 9 closes the loop from "designed" to "running in production": canary releases,
prompt versioning, drift detection, rollback, and containerizing this course's agent
code.